In [1]:
%config IPCompleter.use_jedi = False
import import_ipynb

In [2]:
from pg_01_01_get_enriched_md_files import get_files_from_repo
from pg_02_01_chunking import chunk_everything_by_window, chunk_everything_by_section
from minsearch import Index
from tqdm.auto import tqdm

# Reuse from day 1

In [3]:
OWNER="fastapi"
REPO="typer"
BRANCH="master"

In [4]:
data = [fm_content for fm_content in get_files_from_repo(owner=OWNER, repo=REPO, extensions=[".md", ".mdx"], branch=BRANCH, folder=None)]
len(data)

83

In [5]:
len(data[45]["content"]), data[45].keys()

(12182, dict_keys(['content', 'filename']))

# Reuse from day 2

In [6]:
chunks = chunk_everything_by_window(documents=data)

In [7]:
chunks[0], chunks[0].keys()

({'start': 0,
  'chunk': 'Please read the [Development - Contributing](https://typer.tiangolo.com/contributing/) guidelines in the documentation site.',
  'filename': 'typer-master/contributing.md'},
 dict_keys(['start', 'chunk', 'filename']))

# Indexing: Text (lexical) search

In [8]:
index = Index(text_fields=["chunk", "title", "description", "filename"], keyword_fields=[])
index.fit(chunks)

In [20]:
def text_search(query: str, num_results: int = 5) -> list[any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[any]: A list of up to 5 search results returned by the FAQ index.
    """
    return index.search(query, num_results=num_results)

In [10]:
query = 'Which type annotations can I use for cli arguments and options?'
results = text_search(query=query)
results

[{'start': 5000,
  'chunk': ' but you could do it if you need it.\n\nFor example if you had a callback that could be used by several *CLI parameters*, that way the callback could know which parameter is each time.\n\nCheck it:\n\n<div class="termy">\n\n```console\n$ python main.py --name Camila\n\nValidating param: name\nHello Camila\n```\n\n</div>\n\n## Technical Details\n\nBecause you get the relevant data in the callback function based on standard Python type annotations, you get type checks and autocompletion in your editor for free.\n\nAnd **Typer** will make sure you get the function parameters you want.\n\nYou don\'t have to worry about their names, their order, etc.\n\nAs it\'s based on standard Python types, it "**just works**". ✨\n\n### Callback with type annotations\n\nYou can get the `typer.Context` and the `typer.CallbackParam` simply by declaring a function parameter of each type.\n\nThe order doesn\'t matter, the name of the function parameters doesn\'t matter.\n\nYou co

# Indexing: Vector (Semantical) search

In [11]:
# Turn chunks into embeddings (e.g. sentence-transformers package)
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')
# The multi-qa-distilbert-cos-v1 model is trained explicitly for question-answering tasks. 
# It creates embeddings optimized for finding answers to questions.

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [12]:
# This takes time
import numpy as np
embedded_chunks = []
for chunk in tqdm(chunks):
    v_chunk = embedding_model.encode(chunk["chunk"])
    embedded_chunks.append(v_chunk)

embedded_chunks = np.array(embedded_chunks)

  0%|          | 0/474 [00:00<?, ?it/s]

In [13]:
from minsearch import VectorSearch

typer_index = VectorSearch()
typer_index.fit(embedded_chunks, chunks)

In [22]:
def semantic_search(query: str, num_results: int = 5) -> list[any]:
    """
    Perform a semantic (embedding-based) search on the index.

    Args:
        query (str): The natural language search query.
        num_results (int, optional): Number of results to return. Defaults to 5.

    Returns:
        List[any]: Up to `num_results` semantically similar results.
    """
    embedded_query = embedding_model.encode(query)
    return typer_index.search(embedded_query, num_results=num_results)

In [15]:
results = semantic_search(query=query)
results

[{'start': 0,
  'chunk': "# CLI Arguments\n\nIn the next few sections we'll see some ways to modify how *CLI arguments* work.\n\nWe'll create optional *CLI arguments*, we'll add integrated help for *CLI arguments*, etc.",
  'filename': 'typer-master/docs/tutorial/arguments/index.md'},
 {'start': 83000,
  'chunk': '(https://typer.tiangolo.com/tutorial/arguments/optional/#an-alternative-cli-argument-declaration).\n    * It is no longer required to pass a default value of `...` to mark a *CLI Argument* or *CLI Option* as required.\n    * It is now recommended to use `Annotated` for `typer.Option()` and `typer.Argument()`.\n    * All the docs have been updated to recommend `Annotated`.\n\n### Docs\n\n* 📝 Update docs examples for custom param types using `Annotated`, fix overloads for `typer.Argument`. PR [#594](https://github.com/tiangolo/typer/pull/594) by [@tiangolo](https://github.com/tiangolo).\n\n### Internal\n\n* ⬆ [pre-commit.ci] pre-commit autoupdate. PR [#592](https://github.com/t

# Combined search (text + semantic)

In [23]:
def combined_search(query: str, num_results: int = 5) -> list[any]:
    """
    Perform a hybrid search by combining text-based and semantic search results.

    This function executes both keyword (text) search and embedding-based (semantic)
    search, then merges the results while removing duplicates.

    Args:
        query (str): The search query in natural language.
        num_results (int, optional): Number of results to retrieve per search method.
            Defaults to 5.

    Returns:
        list[Any]: A deduplicated list of results from both search strategies.
    """
    t_results = text_search(query=query, num_results=num_results)
    s_results = semantic_search(query=query, num_results=num_results)

    results = []
    seen = set()
    for r in t_results + s_results:
        key = r["filename"] + str(r["start"])
        if key not in seen:
            seen.add(key)
            results.append(r)
    return results      

In [17]:
r = combined_search(query=query, num_results=3)
r, len(r)

([{'start': 5000,
   'chunk': ' but you could do it if you need it.\n\nFor example if you had a callback that could be used by several *CLI parameters*, that way the callback could know which parameter is each time.\n\nCheck it:\n\n<div class="termy">\n\n```console\n$ python main.py --name Camila\n\nValidating param: name\nHello Camila\n```\n\n</div>\n\n## Technical Details\n\nBecause you get the relevant data in the callback function based on standard Python type annotations, you get type checks and autocompletion in your editor for free.\n\nAnd **Typer** will make sure you get the function parameters you want.\n\nYou don\'t have to worry about their names, their order, etc.\n\nAs it\'s based on standard Python types, it "**just works**". ✨\n\n### Callback with type annotations\n\nYou can get the `typer.Context` and the `typer.CallbackParam` simply by declaring a function parameter of each type.\n\nThe order doesn\'t matter, the name of the function parameters doesn\'t matter.\n\nYou 

In [18]:
r = text_search(query, 3)
r

[{'start': 5000,
  'chunk': ' but you could do it if you need it.\n\nFor example if you had a callback that could be used by several *CLI parameters*, that way the callback could know which parameter is each time.\n\nCheck it:\n\n<div class="termy">\n\n```console\n$ python main.py --name Camila\n\nValidating param: name\nHello Camila\n```\n\n</div>\n\n## Technical Details\n\nBecause you get the relevant data in the callback function based on standard Python type annotations, you get type checks and autocompletion in your editor for free.\n\nAnd **Typer** will make sure you get the function parameters you want.\n\nYou don\'t have to worry about their names, their order, etc.\n\nAs it\'s based on standard Python types, it "**just works**". ✨\n\n### Callback with type annotations\n\nYou can get the `typer.Context` and the `typer.CallbackParam` simply by declaring a function parameter of each type.\n\nThe order doesn\'t matter, the name of the function parameters doesn\'t matter.\n\nYou co

In [19]:
r = semantic_search(query, 3)
r

[{'start': 0,
  'chunk': "# CLI Arguments\n\nIn the next few sections we'll see some ways to modify how *CLI arguments* work.\n\nWe'll create optional *CLI arguments*, we'll add integrated help for *CLI arguments*, etc.",
  'filename': 'typer-master/docs/tutorial/arguments/index.md'},
 {'start': 83000,
  'chunk': '(https://typer.tiangolo.com/tutorial/arguments/optional/#an-alternative-cli-argument-declaration).\n    * It is no longer required to pass a default value of `...` to mark a *CLI Argument* or *CLI Option* as required.\n    * It is now recommended to use `Annotated` for `typer.Option()` and `typer.Argument()`.\n    * All the docs have been updated to recommend `Annotated`.\n\n### Docs\n\n* 📝 Update docs examples for custom param types using `Annotated`, fix overloads for `typer.Argument`. PR [#594](https://github.com/tiangolo/typer/pull/594) by [@tiangolo](https://github.com/tiangolo).\n\n### Internal\n\n* ⬆ [pre-commit.ci] pre-commit autoupdate. PR [#592](https://github.com/t